# Libya Emberger Bioclimatic Zoning — Full Python / Earth Engine version

This notebook is the Python counterpart of the reference Google Earth Engine JavaScript application.

It includes all five analytical scenarios:
1. WorldClim V1 historical reference
2. TerraClimate recent baseline
3. ERA5-Land updated recent zoning
4. TerraClimate + ERA5-Land cross-dataset sensitivity
5. CHIRPS precipitation + ERA5-Land temperature hybrid

It also includes Emberger Q2, fixed or exploratory Libya-relative classification, winter thermal variants, Scenario 4 agreement and class-difference layers, area statistics, interactive maps, and Google Drive GeoTIFF exports.

**Scientific limitation:** these are bioclimatic zones, not complete agro-ecological zones (AEZ). Dataset agreement is sensitivity evidence, not ground validation.


In [ ]:
import ee

PROJECT_ID = "practical-proxy-441422-n6"

ee.Authenticate(auth_mode="notebook", force=True)
ee.Initialize(project=PROJECT_ID)

print("SUCCESS: Earth Engine initialized.")


In [ ]:
from urllib.request import urlretrieve
import sys

ENGINE_URL = (
    "https://raw.githubusercontent.com/hamedsabzchi/"
    "libya-emberger-bioclimate/main/python/bioclimate_engine.py"
)
ENGINE_PATH = "/content/bioclimate_engine.py"

urlretrieve(ENGINE_URL, ENGINE_PATH)

if "bioclimate_engine" in sys.modules:
    del sys.modules["bioclimate_engine"]

import bioclimate_engine as bio

print("Full Bioclimate Python engine loaded.")
print("Available scenarios: 1, 2, 3, 4, 5")


## Interactive control panel

Use the controls below instead of editing code. Scenario 4 is the recommended TerraClimate–ERA5-Land sensitivity comparison. Fixed thresholds are recommended for reproducible comparison. Relative mode creates Libya-specific equal-frequency Q2 classes and must not be interpreted as formal Emberger stages. Enable area diagnostics only when you need the approximate class-area table.


In [ ]:
import ipywidgets as widgets
import pandas as pd
from IPython.display import display, clear_output

scenario = widgets.Dropdown(
    options=[
        ("1 - WorldClim historical reference", 1),
        ("2 - TerraClimate recent baseline", 2),
        ("3 - ERA5-Land updated recent zoning", 3),
        ("4 - TerraClimate + ERA5-Land sensitivity", 4),
        ("5 - CHIRPS + ERA5-Land hybrid", 5),
    ],
    value=4,
    description="Scenario:",
    style={"description_width": "initial"},
)

start_year = widgets.IntText(value=1991, description="Start year:")
end_year = widgets.IntText(value=2020, description="End year:")

classification = widgets.Dropdown(
    options=[
        ("Fixed Q2 thresholds", "fixed"),
        ("Exploratory Libya-relative quantiles", "relative"),
    ],
    value="fixed",
    description="Classification:",
    style={"description_width": "initial"},
)

thresholds = widgets.Text(
    value="20,40,60,100,140",
    description="Q2 thresholds:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)

output = widgets.Dropdown(
    options=[
        ("Bioclimatic zones", "zones"),
        ("Emberger Q2", "q2"),
        ("Annual precipitation", "precipitation"),
        ("Hottest-month Tmax", "tmax"),
        ("Coldest-month Tmin", "tmin"),
        ("Winter thermal variants", "winter"),
        ("Scenario 4 exact agreement", "agreement"),
        ("Scenario 4 absolute class difference", "class_difference"),
        ("Scenario 4 Q2 difference", "q2_difference"),
    ],
    value="zones",
    description="Map output:",
    style={"description_width": "initial"},
)

diagnostics = widgets.Checkbox(
    value=False,
    description="Calculate approximate class-area statistics (20 km diagnostic scale)",
    indent=False,
)

run_button = widgets.Button(
    description="RUN ANALYSIS",
    button_style="success",
    icon="play",
)

analysis_output = widgets.Output()
STATE = {"result": None, "tasks": []}

def _scenario_changed(change):
    s = change["new"]
    if s == 1:
        start_year.disabled = True
        end_year.disabled = True
    else:
        start_year.disabled = False
        end_year.disabled = False
        if s == 2:
            start_year.value, end_year.value = 1991, 2020
        elif s == 3:
            start_year.value, end_year.value = 1991, 2025
        elif s == 4:
            start_year.value, end_year.value = 1991, 2020
        elif s == 5:
            start_year.value, end_year.value = 1991, 2020

scenario.observe(_scenario_changed, names="value")

def _run(_):
    with analysis_output:
        clear_output(wait=True)
        try:
            threshold_values = [float(v.strip()) for v in thresholds.value.split(",")]
            sy = None if scenario.value == 1 else start_year.value
            ey = None if scenario.value == 1 else end_year.value

            print("Building Earth Engine computation...")
            result = bio.run_scenario(
                scenario=scenario.value,
                start_year=sy,
                end_year=ey,
                thresholds=threshold_values,
                classification_mode=classification.value,
            )
            STATE["result"] = result

            print("Analysis graph created successfully.")
            print("Main product:", result["mainProduct"]["label"])
            print("Classification:", classification.value)

            if result.get("warning"):
                print("WARNING:", result["warning"])

            if classification.value == "relative":
                applied = bio.get_applied_thresholds(
                    result["mainProduct"],
                    threshold_values,
                    classification.value,
                )
                print("Applied relative Q2 thresholds:", [round(v, 3) for v in applied])

            print("Rendering map...")
            display(bio.display_map(result, output=output.value))

            if diagnostics.value:
                print("Calculating approximate class areas...")
                records = bio.zone_area_records(result, which="main")
                df = pd.DataFrame(records).sort_values("zone")
                display(df)

            print("DONE.")
        except Exception as exc:
            print("ERROR:", type(exc).__name__, "-", exc)

run_button.on_click(_run)

display(
    widgets.VBox([
        scenario,
        widgets.HBox([start_year, end_year]),
        classification,
        thresholds,
        output,
        diagnostics,
        run_button,
        analysis_output,
    ])
)


## Google Drive exports

First run an analysis above. Then use these buttons. Exports use the same spacing as the reference application: WorldClim 1 km, TerraClimate 4 km, and ERA5-Land / Scenario 4 / CHIRPS–ERA5 hybrid 10 km. The buttons start Earth Engine export tasks to the Google Drive folder `Libya_Emberger_Zoning`.


In [ ]:
export_output = widgets.Output()

export_zones_btn = widgets.Button(description="Export zones", icon="download")
export_q2_btn = widgets.Button(description="Export Q2", icon="download")
export_inputs_btn = widgets.Button(description="Export climate inputs", icon="download")
export_comparison_btn = widgets.Button(description="Export Scenario 4 comparison", icon="download")
check_tasks_btn = widgets.Button(description="Check export status", icon="refresh")

def _start_export(kind):
    with export_output:
        result = STATE.get("result")
        if result is None:
            print("Run an analysis first.")
            return
        try:
            if kind == "zones":
                task = bio.export_zones(result, start=True)
            elif kind == "q2":
                task = bio.export_q2(result, start=True)
            elif kind == "inputs":
                task = bio.export_climate_inputs(result, start=True)
            elif kind == "comparison":
                task = bio.export_comparison(result, start=True)
            else:
                raise ValueError(kind)

            STATE["tasks"].append(task)
            print("Export started:", task.status().get("description"))
            print("Task ID:", task.id)
        except Exception as exc:
            print("ERROR:", type(exc).__name__, "-", exc)

export_zones_btn.on_click(lambda _: _start_export("zones"))
export_q2_btn.on_click(lambda _: _start_export("q2"))
export_inputs_btn.on_click(lambda _: _start_export("inputs"))
export_comparison_btn.on_click(lambda _: _start_export("comparison"))

def _check(_):
    with export_output:
        if not STATE["tasks"]:
            print("No export tasks have been started in this session.")
            return
        for i, task in enumerate(STATE["tasks"], start=1):
            status = task.status()
            print(i, status.get("description"), "->", status.get("state"), status.get("error_message", ""))

check_tasks_btn.on_click(_check)

display(
    widgets.VBox([
        widgets.HBox([export_zones_btn, export_q2_btn]),
        widgets.HBox([export_inputs_btn, export_comparison_btn]),
        check_tasks_btn,
        export_output,
    ])
)


## Interpretation reminder

- Q2 describes climatic moisture background.
- Fixed Q2 thresholds are preliminary working boundaries unless independently validated for Libya.
- Relative classes are exploratory ranks within Libya only.
- Scenario 4 does not average TerraClimate and ERA5-Land; it compares them directly.
- Agreement between gridded datasets is not accuracy.
- Before publication or operational use, validate climate inputs and Q2 against available Libyan stations and relevant published/national evidence.
